In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import logomaker
import pandas as pd
import os

In [ ]:
"""
Load in results from `run_bqp.py` -> script for running variant predictions
- Results for top hits (highest |JSD|) overlapping in both DAM and homeostatic models
"""
DAM_HDF5 = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/DAM/top_bottom_overlap/results.hdf5"
HOM_HDF5 = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/homeostatic/top_bottom_overlap/results.hdf5"
OUT_DIR = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/comparison_plots"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Function for one-hot encodding DNA sequence into numeric values
def one_hot_encode(seq):
    mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
    one_hot = np.zeros((len(seq), 4))
    for i, base in enumerate(seq.upper()):
        if base in mapping:
            one_hot[i, mapping[base]] = 1
    return one_hot

In [ ]:
# Function for plotting base-resolution contribution scores
def plot_seqlogo(ax, shap_scores, one_hot_seq, title, window=40):
    center = shap_scores.shape[0] // 2
    scores = shap_scores[center-window:center+window, :]
    seq    = one_hot_seq[center-window:center+window, :]
    projected = scores * seq
    df = pd.DataFrame(projected, columns=["A", "C", "G", "T"])
    logomaker.Logo(df, ax=ax, color_scheme="classic")
    ax.axvline(x=window, color="lightblue", linewidth=1.5, zorder=5)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Position (bp)")
    ax.set_ylabel("Importance Score")

In [ ]:
# Plotting function for DAM and homeostatic variant predictions and contribution scores
def plot_variant(posid, dam_h5, hom_h5, out_dir):
    with h5py.File(dam_h5, "r") as dam_f, h5py.File(hom_h5, "r") as hom_f:
        if posid not in dam_f or posid not in hom_f:
            print(f"Skipping {posid} - not found in both HDF5 files")
            return

        dam = dam_f[posid]["BQP_models"]["DAM"]
        hom = hom_f[posid]["BQP_models"]["homeostatic"]

        dam_ref_profile = dam["profile_predictions"][0, :]
        dam_alt_profile = dam["profile_predictions"][1, :]
        dam_ref_shap    = dam["shap"][0, :, :]
        dam_alt_shap    = dam["shap"][1, :, :]
        dam_jsd         = float(dam["jsd"][()])

        hom_ref_profile = hom["profile_predictions"][0, :]
        hom_alt_profile = hom["profile_predictions"][1, :]
        hom_ref_shap    = hom["shap"][0, :, :]
        hom_alt_shap    = hom["shap"][1, :, :]
        hom_jsd         = float(hom["jsd"][()])

        ref_seq_str = dam_f[posid]["ref_sequence"][()].decode("utf-8")

    ref_one_hot = one_hot_encode(ref_seq_str)
    alt_one_hot = ref_one_hot.copy()
    center = alt_one_hot.shape[0] // 2
    alt_base = posid.split("_")[-1]
    mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
    alt_one_hot[center, :] = 0
    alt_one_hot[center, mapping[alt_base]] = 1

    trim = 50
    profile_len = len(dam_ref_profile)
    bases = np.arange(trim, profile_len - trim) - profile_len // 2
    dam_ref_trim = dam_ref_profile[trim:-trim]
    dam_alt_trim = dam_alt_profile[trim:-trim]
    hom_ref_trim = hom_ref_profile[trim:-trim]
    hom_alt_trim = hom_alt_profile[trim:-trim]

    fig = plt.figure(figsize=(16, 14))
    fig.suptitle(posid, fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(4, 2, hspace=0.6, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
    ax1.plot(bases, dam_ref_trim, color="#2166AC", label="Reference", linewidth=0.8)
    ax1.plot(bases, dam_alt_trim, color="#FF7F00", label="Alternate", linewidth=0.8)
    ax1.axvline(0, color="lightblue", linewidth=1.5)
    ax1.set_title(f"DAM — Predicted Profile (JSD={dam_jsd:.3f})", fontsize=9, fontweight="bold")
    ax1.set_xlabel("Bases (bp)")
    ax1.set_ylabel("Predicted Counts")
    ax1.legend(fontsize=7)
    ax2.plot(bases, hom_ref_trim, color="#2166AC", label="Reference", linewidth=0.8)
    ax2.plot(bases, hom_alt_trim, color="#FF7F00", label="Alternate", linewidth=0.8)
    ax2.axvline(0, color="lightblue", linewidth=1.5)
    ax2.set_title(f"Homeostatic — Predicted Profile (JSD={hom_jsd:.3f})", fontsize=9, fontweight="bold")
    ax2.set_xlabel("Bases (bp)")
    ax2.set_ylabel("Predicted Counts")
    ax2.legend(fontsize=7)

    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1], sharey=ax3)
    ax5 = fig.add_subplot(gs[2, 0], sharey=ax3)
    ax6 = fig.add_subplot(gs[2, 1], sharey=ax3)
    ax7 = fig.add_subplot(gs[3, 0], sharey=ax3)
    ax8 = fig.add_subplot(gs[3, 1], sharey=ax3)

    plot_seqlogo(ax3, dam_ref_shap, ref_one_hot, "DAM — Reference Importance Scores")
    plot_seqlogo(ax4, hom_ref_shap, ref_one_hot, "Homeostatic — Reference Importance Scores")
    plot_seqlogo(ax5, dam_alt_shap, alt_one_hot, "DAM — Alternate Importance Scores")
    plot_seqlogo(ax6, hom_alt_shap, alt_one_hot, "Homeostatic — Alternate Importance Scores")
    plot_seqlogo(ax7, dam_ref_shap - dam_alt_shap, ref_one_hot, "DAM — Delta Scores (ref - alt)")
    plot_seqlogo(ax8, hom_ref_shap - hom_alt_shap, ref_one_hot, "Homeostatic — Delta Scores (ref - alt)")

    outpath = os.path.join(out_dir, f"{posid}_comparison.pdf")
    plt.savefig(outpath, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"Saved: {posid}")

In [23]:
# Get all variant IDs from DAM HDF5 and loop
with h5py.File(DAM_HDF5, "r") as f:
    all_posids = list(f.keys())

In [24]:
all_posids

['chr10_15515407_C_T',
 'chr10_80520381_T_G',
 'chr11_60065650_T_A',
 'chr11_60251677_C_T',
 'chr11_86151686_A_G',
 'chr12_132481571_G_A',
 'chr12_34303244_C_G',
 'chr14_92489041_A_C',
 'chr16_31072003_G_A',
 'chr16_31102913_A_G',
 'chr17_45451927_G_A',
 'chr17_45661310_C_A',
 'chr17_45760551_T_C',
 'chr17_45890853_A_G',
 'chr17_45988535_C_T',
 'chr17_46005237_A_C',
 'chr17_46071308_A_G',
 'chr17_5132669_C_G',
 'chr17_78426734_A_G',
 'chr17_78428421_C_G',
 'chr19_18462024_C_T',
 'chr19_44721572_G_T',
 'chr1_207612944_A_G',
 'chr1_207631538_T_C',
 'chr2_101686188_T_C',
 'chr2_127072128_T_C',
 'chr2_134782976_A_G',
 'chr2_95323524_A_T',
 'chr3_122458999_C_T',
 'chr3_182994652_T_C',
 'chr3_49067703_A_G',
 'chr3_52307782_G_A',
 'chr3_52772615_T_G',
 'chr3_52814517_T_A',
 'chr4_11030398_C_T',
 'chr4_872935_C_T',
 'chr4_89723239_G_A',
 'chr4_89824091_T_C',
 'chr4_958159_T_C',
 'chr4_960459_G_T',
 'chr5_180201150_G_A',
 'chr6_32379713_C_T',
 'chr6_32473619_G_A',
 'chr6_41232816_A_G',
 'chr6_4

In [25]:
print(f"Plotting {len(all_posids)} variants...")
for posid in all_posids:
    plot_variant(posid, DAM_HDF5, HOM_HDF5, OUT_DIR)

print(f"Done! Plots saved to {OUT_DIR}")

Plotting 51 variants...
Saved: chr10_15515407_C_T
Saved: chr10_80520381_T_G
Saved: chr11_60065650_T_A
Saved: chr11_60251677_C_T
Saved: chr11_86151686_A_G
Saved: chr12_132481571_G_A
Saved: chr12_34303244_C_G
Saved: chr14_92489041_A_C
Saved: chr16_31072003_G_A
Saved: chr16_31102913_A_G
Saved: chr17_45451927_G_A
Saved: chr17_45661310_C_A
Saved: chr17_45760551_T_C
Saved: chr17_45890853_A_G
Saved: chr17_45988535_C_T
Saved: chr17_46005237_A_C
Saved: chr17_46071308_A_G
Saved: chr17_5132669_C_G
Saved: chr17_78426734_A_G
Saved: chr17_78428421_C_G
Saved: chr19_18462024_C_T
Saved: chr19_44721572_G_T
Saved: chr1_207612944_A_G
Saved: chr1_207631538_T_C
Saved: chr2_101686188_T_C
Saved: chr2_127072128_T_C
Saved: chr2_134782976_A_G
Saved: chr2_95323524_A_T
Saved: chr3_122458999_C_T
Saved: chr3_182994652_T_C
Saved: chr3_49067703_A_G
Saved: chr3_52307782_G_A
Saved: chr3_52772615_T_G
Saved: chr3_52814517_T_A
Saved: chr4_11030398_C_T
Saved: chr4_872935_C_T
Saved: chr4_89723239_G_A
Saved: chr4_89824091_T_C

In [ ]:
"""
Load in results from `run_bqp.py` -> script for running variant predictions
- Results for top hits (highest |JSD|) NOT overlapping in DAM and homeostatic models
"""
DAM_nonoverlap_HDF5 = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/DAM/top_bottom_nonoverlap/results.hdf5"
HOM_nonoverlap_HDF5 = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/homeostatic/top_bottom_nonoverlap/results.hdf5"
OUT_nonoverlap_DIR = "/gladstone/corces/lab/users/daley/PD_microglia/VariantEffectPrediction/comparison_plots/nonoverlapping_variants"
os.makedirs(OUT_nonoverlap_DIR, exist_ok=True)

In [25]:
# Get all variant IDs from DAM HDF5 and loop
with h5py.File(DAM_nonoverlap_HDF5, "r") as f:
    all_posids_nonoverlap = list(f.keys())

In [26]:
len(all_posids_nonoverlap)

58

In [27]:
print(f"Plotting {len(all_posids_nonoverlap)} variants...")
for posid in all_posids_nonoverlap:
    plot_variant(posid, DAM_nonoverlap_HDF5, HOM_nonoverlap_HDF5, OUT_nonoverlap_DIR)

print(f"Done! Plots saved to {OUT_nonoverlap_DIR}")

Plotting 58 variants...
Saved: chr10_59980504_A_G
Saved: chr10_80316844_G_A
Saved: chr11_121599937_C_T
Saved: chr11_65879789_A_G
Saved: chr11_86117506_G_A
Saved: chr14_67522380_G_A
Saved: chr16_52592639_A_G
Saved: chr16_52598438_A_C
Saved: chr17_45432821_G_C
Saved: chr17_45748985_A_G
Saved: chr17_45815894_T_C
Saved: chr17_45818287_A_G
Saved: chr17_45848600_G_A
Saved: chr17_45896864_C_T
Saved: chr17_45897110_G_T
Saved: chr17_45944677_A_C
Saved: chr17_46047169_T_C
Saved: chr17_46256686_G_A
Saved: chr19_44732689_G_T
Saved: chr19_45128558_G_A
Saved: chr19_45146580_T_G
Saved: chr19_45152075_T_C
Saved: chr19_45221434_T_C
Saved: chr1_161127451_T_A
Saved: chr1_207331929_C_T
Saved: chr20_56409712_G_T
Saved: chr20_6031288_C_T
Saved: chr2_127094628_G_A
Saved: chr2_233133436_A_G
Saved: chr2_95068102_T_C
Saved: chr2_95322844_C_T
Saved: chr3_151382889_T_G
Saved: chr3_48856347_T_C
Saved: chr3_52557248_G_A
Saved: chr4_113453657_C_T
Saved: chr4_15735725_G_A
Saved: chr4_17986518_A_G
Saved: chr4_89704988